# 🧠 LangChain Tools — Access & Update State
### Using ToolRuntime to Read State & Command to Update State
> **Python 3.11 | LangChain v1.0.0 | .env config**


## Cell 1 — Install & Imports

In [ ]:
%pip install langchain-openai python-dotenv langgraph -q

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.tools import tool, ToolRuntime
from langgraph.prebuilt import create_react_agent
from langgraph.types import Command
from langgraph.graph import MessagesState
from typing import TypedDict
import os
from dotenv import load_dotenv

load_dotenv()
print("✅ Ready!")

## Cell 2 — LLM Setup

In [ ]:
llm = ChatOpenAI(
    model=os.getenv("MODEL"),
    base_url=os.getenv("API_URL"),
    api_key=os.getenv("API_KEY"),
    temperature=0,
)
print("✅ LLM configured!")

## Cell 3 — What is ToolRuntime & Command?

### Access State → use `ToolRuntime`
- `runtime` parameter is injected automatically into the tool
- LLM **never sees it** — it's hidden from the tool schema
- Use `runtime.state["key"]` to read any state field

### Update State → use `Command`
- Return `Command(update={"key": value})` from a tool
- This updates the agent state directly
- Use reducers when multiple tools update the same field in parallel


## Cell 4 — ACCESS State using `runtime.state`

In [ ]:
# ToolRuntime gives the tool access to the full agent state
# The 'runtime' parameter is completely hidden from the LLM

@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """Get the most recent message from the user."""
    messages = runtime.state["messages"]

    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return message.content

    return "No user messages found"


@tool
def get_user_preference(
    pref_name: str,          # LLM sees this
    runtime: ToolRuntime     # LLM does NOT see this
) -> str:
    """Get a user preference value by name."""
    preferences = runtime.state.get("user_preferences", {})
    return preferences.get(pref_name, "Not set")


print("✅ Access tools defined!")
print(f"\n📋 Tool schema visible to LLM:")
print(f"   get_last_user_message → {list(get_last_user_message.args_schema.schema().get('properties', {}).keys())}")
print(f"   get_user_preference   → {list(get_user_preference.args_schema.schema().get('properties', {}).keys())}")
# Notice: 'runtime' does not appear in either!

## Cell 5 — UPDATE State using `Command`

In [ ]:
# Returning Command(update={...}) from a tool
# updates the agent state directly

@tool
def set_user_name(new_name: str) -> Command:
    """Set the user's name in the conversation state."""
    print(f"   ✏️  Updating state: user_name = '{new_name}'")
    return Command(update={"user_name": new_name})


@tool
def set_user_preference(
    pref_name: str,
    pref_value: str,
) -> Command:
    """Set a user preference in the conversation state."""
    print(f"   ✏️  Updating state: user_preferences['{pref_name}'] = '{pref_value}'")
    return Command(update={"user_preferences": {pref_name: pref_value}})


print("✅ Update tools defined!")

## Cell 6 — Define Custom State Schema

In [ ]:
# MessagesState already includes 'messages' with a built-in reducer
# We extend it with our own custom fields

class AgentState(MessagesState):
    user_name        : str
    user_preferences : dict

print("✅ Custom state schema defined!")
print(f"   Fields: {list(AgentState.__annotations__.keys())}") 

## Cell 7 — Create Agent with All Tools

In [ ]:
all_tools = [
    get_last_user_message,
    get_user_preference,
    set_user_name,
    set_user_preference,
]

agent = create_react_agent(
    model=llm,
    tools=all_tools,
    state_schema=AgentState,
    prompt=SystemMessage(content=(
        "You are a helpful assistant. "
        "Use tools to read and update user information when asked."
    ))
)

print("✅ Agent created!")
print(f"   Tools: {[t.name for t in all_tools]}") 

## Cell 8 — Run: ACCESS State (Read Last Message)

In [ ]:
response = agent.invoke({
    "messages": [
        HumanMessage(content="Hello! I love Python programming."),
        HumanMessage(content="What was my last message?"),
    ],
    "user_name"        : "",
    "user_preferences" : {},
})

print("🤖", response["messages"][-1].content)

## Cell 9 — Run: ACCESS Custom State Field (Preference)

In [ ]:
response = agent.invoke({
    "messages": [
        HumanMessage(content="What is my theme preference?"),
    ],
    "user_name"        : "",
    "user_preferences" : {"theme": "dark", "language": "Python"},
})

print("🤖", response["messages"][-1].content)

## Cell 10 — Run: UPDATE State (Set User Name)

In [ ]:
response = agent.invoke({
    "messages": [
        HumanMessage(content="My name is Priya. Please remember it."),
    ],
    "user_name"        : "",
    "user_preferences" : {},
})

print("🤖", response["messages"][-1].content)
print(f"\n📌 Updated State:")
print(f"   user_name → '{response['user_name']}'")

## Cell 11 — Run: UPDATE Custom Preference Field

In [ ]:
response = agent.invoke({
    "messages": [
        HumanMessage(content="Set my theme preference to light mode."),
    ],
    "user_name"        : "Priya",
    "user_preferences" : {},
})

print("🤖", response["messages"][-1].content)
print(f"\n📌 Updated State:")
print(f"   user_preferences → {response['user_preferences']}") 

## ✅ Summary

### Access State
```python
from langchain.tools import tool, ToolRuntime

@tool
def my_tool(runtime: ToolRuntime) -> str:
    data = runtime.state["messages"]            # read messages
    pref = runtime.state.get("user_preferences", {})  # read custom field
```
→ `runtime` is **hidden from LLM** automatically

---

### Update State
```python
from langgraph.types import Command

@tool
def my_tool(value: str) -> Command:
    return Command(update={"field_name": value})  # update any state field
```
→ updates agent state directly

---

| Concept      | Tool          | How                              |
|--------------|---------------|----------------------------------|
| Read state   | ToolRuntime   | `runtime.state["key"]`           |
| Write state  | Command       | `Command(update={"key": value})` |
| Parallel safe| Reducer       | Resolves conflicts on same field |
